In [1]:
!pip install gensim nltk numpy

import nltk
from nltk.corpus import brown
import gensim
from gensim.models import Word2Vec, FastText
import numpy as np
import warnings
warnings.filterwarnings('ignore')

nltk.download('brown')
print("Setup complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 32.5 MB/s eta 0:00:00


[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.


Setup complete!


In [4]:
brown_sentences = brown.sents()
brown_sentences = [[word.lower() for word in sent] for sent in brown_sentences]

print(f"Number of Brown sentences: {len(brown_sentences)}")
print("Example sentence:", brown_sentences[0])


Number of Brown sentences: 57340
Example sentence: ['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', "atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that', 'any', 'irregularities', 'took', 'place', '.']


In [5]:
extra_sentences_str = [
    "the teacher is teaching students about grammar",
    "she teaches mathematics at the university",
    "the teacher prepared teaching materials yesterday",
    "many teachers attend teaching conferences annually",
    "effective teaching requires good communication skills",

    "the unteachable student refused to learn",

    "computational linguistics combines computer science and language",
    "the computation took several hours to complete",
    "we computed the results using advanced algorithms",
    "modern computers can compute complex calculations quickly",

    "the recomputation was necessary after finding errors",

    "natural language processing is fascinating",
    "the nature of language is complex",
    "naturally occurring patterns in text are important",

    "the unnaturalness of the translation was obvious",
]

extra_sentences = [sent.lower().split() for sent in extra_sentences_str]

print(f"Number of extra sentences: {len(extra_sentences)}")
print("Example extra sentence:", extra_sentences[0])


Number of extra sentences: 15
Example extra sentence: ['the', 'teacher', 'is', 'teaching', 'students', 'about', 'grammar']


In [6]:
all_sentences = brown_sentences + extra_sentences
print(f"Total number of sentences in training data: {len(all_sentences)}")


Total number of sentences in training data: 57355


In [7]:
w2v_model = Word2Vec(
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    workers=4,
    seed=42
)

w2v_model.build_vocab(all_sentences)
print(f"Word2Vec vocabulary size: {len(w2v_model.wv)}")

w2v_model.train(
    all_sentences,
    total_examples=w2v_model.corpus_count,
    epochs=10
)

print(" Word2Vec training finished.")


Word2Vec vocabulary size: 49818
 Word2Vec training finished.


In [8]:
w2v_model.save("brown_word2vec.model")
print("Saved Word2Vec model as 'brown_word2vec.model'")


Saved Word2Vec model as 'brown_word2vec.model'


In [9]:
ft_model = FastText(
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    workers=4,
    seed=42,
    min_n=3,
    max_n=6
)

ft_model.build_vocab(all_sentences)
print(f"FastText vocabulary size: {len(ft_model.wv)}")

ft_model.train(
    all_sentences,
    total_examples=ft_model.corpus_count,
    epochs=10
)

print("FastText training finished.")


FastText vocabulary size: 49818
FastText training finished.


In [10]:
ft_model.save("brown_fasttext.model")
print("Saved FastText model as 'brown_fasttext.model'")


Saved FastText model as 'brown_fasttext.model'


In [11]:
oov_words = [
    "teachable",
    "unteacher",
    "supercomputer",
    "miscomputation",
    "unnaturally"
]

print("Experiment 1: OOV Words\n")

for word in oov_words:
    print(f"Word: '{word}'")

    try:
        vec_w2v = w2v_model.wv[word]
        print("  Word2Vec: vector found (OOV handled - but this should normally fail if truly OOV)")
    except KeyError:
        print("  Word2Vec: KeyError (cannot handle OOV word)")

    try:
        vec_ft = ft_model.wv[word]
        print("  FastText: vector found (uses character n-grams for OOV)")
    except KeyError:
        print("  FastText: KeyError (unexpected for FastText)")

    print()


Experiment 1: OOV Words

Word: 'teachable'
  Word2Vec: KeyError (cannot handle OOV word)
  FastText: vector found (uses character n-grams for OOV)

Word: 'unteacher'
  Word2Vec: KeyError (cannot handle OOV word)
  FastText: vector found (uses character n-grams for OOV)

Word: 'supercomputer'
  Word2Vec: KeyError (cannot handle OOV word)
  FastText: vector found (uses character n-grams for OOV)

Word: 'miscomputation'
  Word2Vec: KeyError (cannot handle OOV word)
  FastText: vector found (uses character n-grams for OOV)

Word: 'unnaturally'
  Word2Vec: vector found (OOV handled - but this should normally fail if truly OOV)
  FastText: vector found (uses character n-grams for OOV)



In [13]:
rare_common_pairs = [
    ("unteachable", "teacher"),
    ("recomputation", "computation"),
    ("unnaturalness", "natural"),
]

print("Experiment 2: Rare Words Quality \n")

for rare, common in rare_common_pairs:
    print(f"Pair: ('{rare}', '{common}')")

    if rare in w2v_model.wv.key_to_index and common in w2v_model.wv.key_to_index:
        sim_w2v = w2v_model.wv.similarity(rare, common)
        print(f"  Word2Vec similarity:   {sim_w2v:.4f}")
    else:
        print("  Word2Vec similarity:   one of the words is missing from vocab")

    if rare in ft_model.wv.key_to_index and common in ft_model.wv.key_to_index:
        sim_ft = ft_model.wv.similarity(rare, common)
        print(f"  FastText similarity:   {sim_ft:.4f}")
    else:
        print("  FastText similarity:   one of the words is missing from vocab")

    print()


Experiment 2: Rare Words Quality 

Pair: ('unteachable', 'teacher')
  Word2Vec similarity:   0.6578
  FastText similarity:   0.6537

Pair: ('recomputation', 'computation')
  Word2Vec similarity:   0.8476
  FastText similarity:   0.9675

Pair: ('unnaturalness', 'natural')
  Word2Vec similarity:   0.6028
  FastText similarity:   0.8381



In [14]:
def print_top_similar(model, word, topn=5, model_name="model"):
    print(f"Top {topn} most similar to '{word}' in {model_name}:")
    if word not in model.wv.key_to_index:
        print(f"  '{word}' not in vocabulary.\n")
        return
    for i, (w, score) in enumerate(model.wv.most_similar(word, topn=topn), start=1):
        print(f"  {i}. {w:15s}  {score:.4f}")
    print()


print("Experiment 3: Morphological Families \n")

print("Family 1: teach / teacher / teaching / teaches\n")

print_top_similar(w2v_model, "teaching", model_name="Word2Vec")
print_top_similar(ft_model, "teaching", model_name="FastText")

print("Family 2: compute / computer / computation / computed / computational\n")

print_top_similar(w2v_model, "computation", model_name="Word2Vec")
print_top_similar(ft_model, "computation", model_name="FastText")


Experiment 3: Morphological Families 

Family 1: teach / teacher / teaching / teaches

Top 5 most similar to 'teaching' in Word2Vec:
  1. counseling       0.8244
  2. testing          0.8071
  3. membership       0.8055
  4. training         0.8042
  5. scholarship      0.8012

Top 5 most similar to 'teaching' in FastText:
  1. aching           0.9286
  2. sky-reaching     0.9113
  3. out-reaching     0.9111
  4. far-reaching     0.9048
  5. preaching        0.9034

Family 2: compute / computer / computation / computed / computational

Top 5 most similar to 'computation' in Word2Vec:
  1. allowable        0.9169
  2. 93               0.9137
  3. non-itemized     0.9132
  4. extrema          0.9126
  5. 7%               0.9121

Top 5 most similar to 'computation' in FastText:
  1. compilation      0.9703
  2. recomputation    0.9675
  3. co-optation      0.9483
  4. deviation        0.9406
  5. dilatation       0.9389



In [15]:
print("Experiment 4: Morphological Analogies \n")

def analogy(model, pos_words, neg_words, model_name="model", topn=3):
    try:
        result = model.wv.most_similar(positive=pos_words, negative=neg_words, topn=topn)
        print(f"{model_name} analogy {pos_words} - {neg_words}:")
        for i, (w, score) in enumerate(result, start=1):
            print(f"  {i}. {w:15s}  {score:.4f}")
        print()
    except KeyError as e:
        print(f"{model_name}: KeyError - missing word {e}\n")


print("Analogy 1: teacher : teaching = computer : ?\n")
analogy(
    w2v_model,
    pos_words=["teaching", "computer"],
    neg_words=["teacher"],
    model_name="Word2Vec"
)

analogy(
    ft_model,
    pos_words=["teaching", "computer"],
    neg_words=["teacher"],
    model_name="FastText"
)

print("Analogy 2: natural : naturally = quick : ?\n")
analogy(
    w2v_model,
    pos_words=["naturally", "quick"],
    neg_words=["natural"],
    model_name="Word2Vec"
)

analogy(
    ft_model,
    pos_words=["naturally", "quick"],
    neg_words=["natural"],
    model_name="FastText"
)


Experiment 4: Morphological Analogies 

Analogy 1: teacher : teaching = computer : ?

Word2Vec analogy ['teaching', 'computer'] - ['teacher']:
  1. ultraviolet      0.7931
  2. developmental    0.7916
  3. ultrasonic       0.7852

FastText analogy ['teaching', 'computer'] - ['teacher']:
  1. computing        0.8823
  2. compiling        0.8298
  3. composing        0.8188

Analogy 2: natural : naturally = quick : ?

Word2Vec analogy ['naturally', 'quick'] - ['natural']:
  1. nadine           0.6712
  2. smoothly         0.6589
  3. sadly            0.6499

FastText analogy ['naturally', 'quick'] - ['natural']:
  1. quickly          0.8654
  2. sickly           0.8219
  3. woolly           0.8145

